# Deep Hedging — Phase 4, brique 4 : Heston AVEC coûts

On combine les **deux frictions** : marché incomplet (Heston) **et** coûts de transaction. C'est le cas le plus réaliste, et le plus dur.

La couverture classique a besoin de deux béquilles séparées : le delta de Black-Scholes à la vol implicite (qui ignore le vega), et une bande de non-transaction ajustée à la main pour les coûts. Le **deep hedger** apprend tout d'un coup : la correction vega ET la bande, conjointement.

**Cibles** (coût=1%, n=63) : delta pur = 9.39, bande optimisée = **8.05**. Le réseau doit battre 8.05.

In [ ]:
import torch
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
torch.manual_seed(0)

## Paramètres, prime, benchmark classique (bande optimisée, numpy)

In [ ]:
S0, K, mu, r, T = 100., 100., 0.05, 0.02, 1.0
v0, kappa, theta, xi, rho = 0.04, 2.0, 0.04, 0.3, -0.7
n, cost, alpha = 63, 0.01, 0.95
dt = T/n
rng = np.random.default_rng(0)

def sim_heston_np(S0, v0, drift, T, n, m):
    dt = T/n; S=np.empty((m,n+1)); v=np.empty((m,n+1)); S[:,0]=S0; v[:,0]=v0
    for k in range(n):
        Z1=rng.standard_normal(m); Z2=rho*Z1+np.sqrt(1-rho**2)*rng.standard_normal(m)
        vk=np.maximum(v[:,k],0.0)
        v[:,k+1]=np.maximum(v[:,k]+kappa*(theta-vk)*dt+xi*np.sqrt(vk)*np.sqrt(dt)*Z2,0.0)
        S[:,k+1]=S[:,k]*np.exp((drift-0.5*vk)*dt+np.sqrt(vk)*np.sqrt(dt)*Z1)
    return S
def bs_price_np(S,K,tau,r,s):
    S=np.asarray(S,float); d1=(np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)); d2=d1-s*np.sqrt(tau)
    return S*norm.cdf(d1)-K*np.exp(-r*tau)*norm.cdf(d2)
def bs_delta_np(S,K,tau,r,s):
    S=np.asarray(S,float); return norm.cdf((np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)))
def cvar_np(pnl,a=0.95):
    loss=-pnl; return loss[loss>=np.quantile(loss,a)].mean()

premium = float(np.exp(-r*T)*np.mean(np.maximum(sim_heston_np(S0,v0,r,T,252,80_000)[:,-1]-K,0)))
sig_imp = brentq(lambda s: bs_price_np(S0,K,T,r,s)-premium, 1e-3, 2.0)
print(f"prime (prix Heston) = {premium:.3f}   vol implicite = {sig_imp:.4f}")

def band_hedge_np(S, band):
    m,n1=S.shape; nn=n1-1; times=np.linspace(0,T,n1); cash=np.full(m,premium); pos=np.zeros(m)
    for k in range(nn):
        tau=T-times[k]; tgt=bs_delta_np(S[:,k],K,tau,r,sig_imp); tr=np.where(np.abs(tgt-pos)>band,tgt-pos,0.0)
        cash-=tr*S[:,k]; cash-=cost*np.abs(tr)*S[:,k]; pos=pos+tr; cash*=np.exp(r*dt)
    return cash+pos*S[:,-1]-np.maximum(S[:,-1]-K,0.0)

SH = sim_heston_np(S0,v0,mu,T,n,80_000)
bands=[0.0,0.06,0.13,0.18,0.25]
cs=[cvar_np(band_hedge_np(SH,b)) for b in bands]
i=int(np.argmin(cs))
print("delta pur (band=0) CVaR =", round(cs[0],3))
print(f"bande optimale = {bands[i]:.2f}, CVaR = {cs[i]:.3f}")

## Le deep hedger : état augmenté (avec v) + coûts dans la perte

In [ ]:
def heston_step_t(S, v, drift, Z1, Z2):
    vk = torch.clamp(v, min=0.0)
    return (S*torch.exp((drift-0.5*vk)*dt + torch.sqrt(vk)*np.sqrt(dt)*Z1),
            torch.clamp(v + kappa*(theta-vk)*dt + xi*torch.sqrt(vk)*np.sqrt(dt)*Z2, min=0.0))

def cvar_ru(loss, w, a=0.95):
    return w + torch.mean(torch.relu(loss - w))/(1.0 - a)

class HedgeNet(torch.nn.Module):
    def __init__(self, h=32):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(4,h),torch.nn.ReLU(),
                                       torch.nn.Linear(h,h),torch.nn.ReLU(),torch.nn.Linear(h,1))
    def forward(self,x): return self.net(x).squeeze(-1)

def hedging_loss(net, w, m):
    S=torch.full((m,),S0); v=torch.full((m,),v0); cash=torch.full((m,),premium); dprev=torch.zeros(m)
    for k in range(n):
        tau=float(T-k*dt)
        feat=torch.stack([torch.log(S/K), torch.full((m,),tau), dprev, v], dim=1)
        dk=net(feat); tr=dk-dprev
        cash = cash - tr*S - cost*torch.abs(tr)*S          # <-- coûts ici
        Z1=torch.randn(m); Z2=rho*Z1+np.sqrt(1-rho**2)*torch.randn(m)
        S,v=heston_step_t(S,v,mu,Z1,Z2); cash=cash*np.exp(r*dt); dprev=dk
    pnl=cash+dprev*S-torch.clamp(S-K,min=0.0)
    return cvar_ru(-pnl, w, alpha)

## Entraînement

In [ ]:
net=HedgeNet(); w=torch.zeros(1,requires_grad=True)
opt=torch.optim.Adam(list(net.parameters())+[w], lr=1e-3)
for it in range(3000):
    opt.zero_grad(); loss=hedging_loss(net,w,2048); loss.backward(); opt.step()
    if it%300==0: print(f"iter {it:4d}  CVaR(train)={loss.item():.3f}")

## Évaluation : réseau vs delta pur vs bande optimisée

In [ ]:
def bs_delta_t(S,K,tau,r,s):
    d1=(torch.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)); return 0.5*(1+torch.erf(d1/np.sqrt(2)))

def rollout(policy, m):
    S=torch.full((m,),S0); v=torch.full((m,),v0); cash=torch.full((m,),premium); dprev=torch.zeros(m)
    for k in range(n):
        tau=float(T-k*dt); dk=policy(S,v,tau,dprev); tr=dk-dprev
        cash=cash-tr*S-cost*torch.abs(tr)*S
        Z1=torch.randn(m); Z2=rho*Z1+np.sqrt(1-rho**2)*torch.randn(m)
        S,v=heston_step_t(S,v,mu,Z1,Z2); cash=cash*np.exp(r*dt); dprev=dk
    return cash+dprev*S-torch.clamp(S-K,min=0.0)
def cvar_t(pnl,a=0.95):
    loss=-pnl; return loss[loss>=torch.quantile(loss,a)].mean().item()

net.eval()
with torch.no_grad():
    m=80_000
    pol_net = lambda S,v,tau,dp: net(torch.stack([torch.log(S/K),torch.full((len(S),),tau),dp,v],dim=1))
    pol_del = lambda S,v,tau,dp: bs_delta_t(S,K,tau,r,sig_imp)   # delta pur (band=0)
    print(f"CVaR réseau        = {cvar_t(rollout(pol_net,m)):.3f}")
    print(f"CVaR delta pur     = {cvar_t(rollout(pol_del,m)):.3f}   (~9.4)")
    print(f"CVaR bande optim.  ~ 8.05  (benchmark numpy ci-dessus)")

## Ce qu'on regarde

Si le réseau passe sous 8.05, il bat le meilleur classique en gérant **conjointement** l'incomplétude et les coûts, là où l'approche classique jongle avec deux heuristiques séparées. On analysera ensuite comment il combine bande et sensibilité à la volatilité. Étape 2 : ajouter des calls d'autres maturités comme instruments pour couvrir le vega, et faire chuter la CVaR encore plus bas.